In [1]:
import sys
import os

In [2]:
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")
print(f"Virtual environment: {os.environ.get('VIRTUAL_ENV', 'Not in a virtual environment')}")

Python executable: /Users/chris/Applications/build.one/.venv/bin/python
Python version: 3.9.13 (v3.9.13:6de2ca5339, May 17 2022, 11:37:23) 
[Clang 13.0.0 (clang-1300.0.29.30)]
Virtual environment: /Users/chris/Applications/build.one/.venv


In [3]:
# Check current directory and add project root to path
print(f"Current working directory: {os.getcwd()}")

Current working directory: /Users/chris/Applications/build.one/tests


In [4]:
# Add the project root to Python path
project_root = '/Users/chris/Applications/build.one'
sys.path.append(project_root)

In [5]:
print(f"Added project root: {project_root}")
print(f"Project exists: {os.path.exists(project_root)}")

Added project root: /Users/chris/Applications/build.one
Project exists: True


In [6]:
# Check if persistence folder exists
persistence_path = os.path.join(project_root, 'persistence')
print(f"Persistence path: {persistence_path}")
print(f"Persistence exists: {os.path.exists(persistence_path)}")

Persistence path: /Users/chris/Applications/build.one/persistence
Persistence exists: True


In [7]:
if os.path.exists(persistence_path):
    print(f"Files in persistence: {os.listdir(persistence_path)}")

Files in persistence: ['pers_project_file.py', 'pers_project_folder.py', 'pers_transaction.py', '__pycache__', 'pers_database.py', 'pers_response.py', 'pers_certificate_of_insurance.py']


In [8]:
# Try to import
try:
    from persistence import pers_database
    print("✓ Successfully imported pers_database")
except ImportError as e:
    print(f"✗ Import failed: {e}")
    print(f"Python path:")
    for i, path in enumerate(sys.path):
        print(f"  {i}: {path}")

✓ Successfully imported pers_database


In [9]:
cnxn = pers_database.get_db_connection()
cnxn

In [10]:
from modules.sub_cost_code import pers_sub_cost_code

In [11]:
db_sub_cost_codes = pers_sub_cost_code.read_sub_cost_codes().data
db_sub_cost_codes

[SubCostCode(id=797, guid='6A96DBA5-42C3-4657-A09B-F0488FBA9392', created_datetime='2022-04-13 13:35:29.0000000 -07:00', modified_datetime='2024-05-14 05:46:17.0000000 -07:00', number=Decimal('1.00'), name='Permits', description=None, cost_code_id=3, transaction_id=1791),
 SubCostCode(id=798, guid='C9B68C73-5CA2-4F60-9E51-9CE0D16E750A', created_datetime='2022-04-13 13:41:27.0000000 -07:00', modified_datetime='2024-05-14 05:44:13.0000000 -07:00', number=Decimal('1.20'), name='Building Permit', description=None, cost_code_id=3, transaction_id=1792),
 SubCostCode(id=799, guid='2522DF67-594B-4112-A97F-49C43167F74D', created_datetime='2022-04-13 14:32:05.0000000 -07:00', modified_datetime='2024-05-14 05:46:23.0000000 -07:00', number=Decimal('1.30'), name='Stormwater Permit', description=None, cost_code_id=3, transaction_id=1793),
 SubCostCode(id=800, guid='9CA38D22-2337-45BD-AD0B-B4705A0921B0', created_datetime='2022-04-13 14:32:28.0000000 -07:00', modified_datetime='2024-05-14 05:46:24.000

In [12]:
import csv

In [15]:
filename='sub_cost_codes_update.csv'

In [ ]:

with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['id', 'guid', 'current_number', 'new_number', 'name', 'cost_code_id', 'notes']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    
    writer.writeheader()
    
    for sub_cost_code in db_sub_cost_codes:
        writer.writerow({
            'id': sub_cost_code.id,
            'guid': sub_cost_code.guid,
            'current_number': str(sub_cost_code.number),
            'new_number': str(sub_cost_code.number),  # Start with same value
            'name': sub_cost_code.name,
            'cost_code_id': sub_cost_code.cost_code_id,
            'notes': ''  # For your manual notes
        })

print(f"Exported {len(db_sub_cost_codes)} records to {filename}")
print("Please edit the 'new_number' column in the CSV file, then run the import function.")

In [13]:
from decimal import Decimal
from datetime import datetime

In [16]:
changes = []
    
with open(filename, 'r', newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)
    
    for row in reader:
        current_number = Decimal(row['current_number'])
        new_number = Decimal(row['new_number'])
        
        if current_number != new_number:
            changes.append({
                'id': int(row['id']),
                'guid': row['guid'],
                'current_number': current_number,
                'new_number': new_number,
                'name': row['name'],
                'cost_code_id': int(row['cost_code_id']),
                'notes': row['notes']
            })

print(f"Found {len(changes)} records to update:")
for change in changes:
    print(f"ID {change['id']}: {change['current_number']} -> {change['new_number']} ({change['name']})")

Found 340 records to update:
ID 798: 1.20 -> 1.02 (Building Permit)
ID 799: 1.30 -> 1.03 (Stormwater Permit)
ID 800: 1.40 -> 1.04 (Water & Sewer Permit)
ID 801: 2.10 -> 2.01 (Dumpsters)
ID 1091: 2.20 -> 2.02 (Dumpsters - Drop Fee)
ID 1092: 2.30 -> 2.03 (Dumpsters - Excess Tonnage)
ID 1093: 2.40 -> 2.04 (Dumpsters - Per Day Rental)
ID 1095: 3.10 -> 3.01 (Portable Toilets - DropFee)
ID 1096: 3.20 -> 3.02 (Portable Toilets - Monthly)
ID 1098: 4.10 -> 4.01 (Surveyor)
ID 1099: 4.20 -> 4.02 (Engineer)
ID 1100: 4.30 -> 4.03 (Architect)
ID 1101: 4.40 -> 4.04 (Designer)
ID 1102: 4.50 -> 4.05 (Landscape Architect)
ID 1103: 4.60 -> 4.06 (Soil Engineer)
ID 1105: 5.10 -> 5.01 (Builder's Risk Policy)
ID 1106: 5.20 -> 5.02 (Liability Insurance)
ID 1108: 6.10 -> 6.01 (Site Prep, Erosion & Entr.)
ID 1109: 6.20 -> 6.02 (Tree Removal)
ID 1110: 6.30 -> 6.03 (Tree Protection)
ID 1111: 6.40 -> 6.04 (Wetlands/Stream Protection)
ID 1112: 6.50 -> 6.05 (Septic/Leach Field Protect)
ID 1113: 6.60 -> 6.06 (Rough G

In [17]:
updated_count = 0
failed_count = 0

for change in changes:
    try:
        # Get the current record
        response = pers_sub_cost_code.read_sub_cost_code_by_id(change['id'])
        
        if response.success:
            sub_cost_code = response.data
            sub_cost_code.number = change['new_number']
            sub_cost_code.modified_datetime = datetime.now()

            if sub_cost_code.description == None:
                sub_cost_code.description = ''
            
            #print(sub_cost_code)
            # Update the record
            update_response = pers_sub_cost_code.update_sub_cost_code(sub_cost_code)
            
            if update_response.success:
                print(f"✓ Updated ID {change['id']}: {change['current_number']} -> {change['new_number']}")
                updated_count += 1
            else:
                print(f"✗ Failed to update ID {change['id']}: {update_response.message}")
                failed_count += 1
        else:
            print(f"✗ Failed to read ID {change['id']}: {response.message}")
            failed_count += 1
            
    except Exception as e:
        print(f"✗ Error updating ID {change['id']}: {str(e)}")
        failed_count += 1

print(f"\nUpdate complete: {updated_count} successful, {failed_count} failed")


✓ Updated ID 798: 1.20 -> 1.02
✓ Updated ID 799: 1.30 -> 1.03
✓ Updated ID 800: 1.40 -> 1.04
✓ Updated ID 801: 2.10 -> 2.01
✓ Updated ID 1091: 2.20 -> 2.02
✓ Updated ID 1092: 2.30 -> 2.03
✓ Updated ID 1093: 2.40 -> 2.04
✓ Updated ID 1095: 3.10 -> 3.01
✓ Updated ID 1096: 3.20 -> 3.02
✓ Updated ID 1098: 4.10 -> 4.01
✓ Updated ID 1099: 4.20 -> 4.02
✓ Updated ID 1100: 4.30 -> 4.03
✓ Updated ID 1101: 4.40 -> 4.04
✓ Updated ID 1102: 4.50 -> 4.05
✓ Updated ID 1103: 4.60 -> 4.06
✓ Updated ID 1105: 5.10 -> 5.01
✓ Updated ID 1106: 5.20 -> 5.02
✓ Updated ID 1108: 6.10 -> 6.01
✓ Updated ID 1109: 6.20 -> 6.02
✓ Updated ID 1110: 6.30 -> 6.03
✓ Updated ID 1111: 6.40 -> 6.04
✓ Updated ID 1112: 6.50 -> 6.05
✓ Updated ID 1113: 6.60 -> 6.06
✓ Updated ID 1114: 6.70 -> 6.07
✓ Updated ID 1115: 6.80 -> 6.08
✓ Updated ID 1116: 6.90 -> 6.09
✓ Updated ID 1128: 7.10 -> 7.01
✓ Updated ID 1129: 7.20 -> 7.02
✓ Updated ID 1130: 7.30 -> 7.03
✓ Updated ID 1131: 7.40 -> 7.04
✓ Updated ID 1132: 7.50 -> 7.05
✓ Updated ID